# Premier League — Previsão de Resultados

**Objetivo:** construir um pipeline de dados para prever o resultado de partidas da Premier League em três classes — vitória do mandante (*HomeWin*), empate (*Draw*) e vitória do visitante (*AwayWin*).

---

### Fonte dos dados
Arquivo `premier_league_all_seasons.csv` com todas as temporadas da Premier League. Utilizamos apenas a janela **2005/06 em diante**, pois as temporadas anteriores têm estatísticas de jogo (chutes, escanteios, cartões) incompletas.

| Recorte | Jogos | Home Win | Draw | Away Win |
|---------|------:|--------:|-----:|---------:|
| 2005/06 → 2025/26 | 7 980 | 45,6 % | 24,3 % | 30,1 % |

---

### Estrutura do notebook

| Seção | Conteúdo |
|-------|----------|
| 1 | Imports e configuração do ambiente |
| 2 | Carga e filtragem do CSV |
| 3 | Limpeza e padronização das colunas |
| 4 | Split temporal treino / teste |
| 6 | Feature engineering (`build_features`) |
| 7 | Elo ratings (`build_elo`) |

## Seção 1 — Imports e configuração

Carregamento das bibliotecas utilizadas ao longo do notebook.

`MPLBACKEND=agg` é definido antes dos imports gráficos para forçar o backend não-interativo do Matplotlib, evitando erros em ambientes sem display (servidores, CI).

In [1]:
import os
os.environ["MPLBACKEND"] = "agg"

import pandas as pd
import numpy as np
import pyarrow
import seaborn as sns
import matplotlib.pyplot as plt

print("Dependências carregadas com sucesso.")

Dependências carregadas com sucesso.


## Seção 2 — Carregar e filtrar

Leitura do CSV bruto e filtragem pelo critério de qualidade de dados.

> **Por que 2005/06?** Temporadas anteriores têm colunas de estatísticas (`HS`, `AS`, `HST`, `AST`, `HC`, `AC`, `HY`, `AY`) com valores nulos sistemáticos, tornando-as inutilizáveis para feature engineering. A partir de 2005/06 todas as colunas essenciais estão completas.

Um `assert` garante que exatamente **7 980 jogos** são carregados — qualquer alteração no CSV ou na lógica de filtro falha ruidosamente aqui.

In [ ]:
df_raw = pd.read_csv("premier_league_all_seasons.csv")

df = df_raw[df_raw["Season"] >= "2005/06"].copy()

assert len(df) == 7980, f"Esperado 7980 jogos, encontrado {len(df)}"
print(f"Jogos carregados: {len(df)}")
df.head(3)

Jogos carregados: 7980


,Season,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,...,HC,AC,HY,AY,HR,AR,Referee,B365H,B365D,B365A
4634,2005/06,2005-08-13,West Ham,Blackburn,3.0,1.0,H,0.0,1.0,A,...,2.0,6.0,0.0,1.0,0.0,1.0,A Wiley,2.50,3.2,2.75
4635,2005/06,2005-08-13,Sunderland,Charlton,1.0,3.0,A,1.0,1.0,D,...,5.0,5.0,2.0,0.0,0.0,1.0,H Webb,2.37,3.2,2.90
4636,2005/06,2005-08-13,Portsmouth,Tottenham,0.0,2.0,A,0.0,1.0,A,...,7.0,2.0,0.0,2.0,0.0,0.0,B Knight,2.75,3.2,2.50


## Seção 3 — Limpeza e padronização

Transformações aplicadas para deixar o DataFrame pronto para feature engineering:

| Operação | Detalhe |
|----------|---------|
| `Date` → `datetime` | Permite ordenação e cortes temporais precisos |
| `season` como inteiro | `"2005/06"` → `2005` — facilita filtros numéricos |
| Renomear colunas | `FTHG/FTAG` → `home_score/away_score`, etc. — nomes autodescritivos |
| `FTR` → `result` | `H/D/A` → `HomeWin/Draw/AwayWin` — legível e sem ambiguidade |
| Ordenar por `Date` | Ordem cronológica é pré-requisito para todo o pipeline de features |

In [3]:
# Converter Date para datetime
df["Date"] = pd.to_datetime(df["Date"])

# Extrair ano inicial da temporada: "2005/06" → 2005
df["season"] = df["Season"].str[:4].astype(int)

# Renomear colunas
df = df.rename(columns={
    "FTHG": "home_score",
    "FTAG": "away_score",
    "HomeTeam": "home_team",
    "AwayTeam": "away_team",
})

# Mapear FTR → result
result_map = {"H": "HomeWin", "D": "Draw", "A": "AwayWin"}
df["result"] = df["FTR"].map(result_map)

# Ordenar por data e resetar index
df = df.sort_values("Date").reset_index(drop=True)

print(f"Shape: {df.shape}")
print(f"Nulos em result: {df['result'].isna().sum()}")
print(f"\nDistribuição de resultados:\n{df['result'].value_counts(normalize=True).round(3)}")
df[["Date", "season", "home_team", "away_team", "home_score", "away_score", "result"]].head(5)

Shape: (7980, 28)
Nulos em result: 0

Distribuição de resultados:
result
HomeWin    0.456
AwayWin    0.301
Draw       0.243
Name: proportion, dtype: float64


,Date,season,home_team,away_team,home_score,away_score,result
0,2005-08-13,2005,West Ham,Blackburn,3.0,1.0,HomeWin
1,2005-08-13,2005,Sunderland,Charlton,1.0,3.0,AwayWin
2,2005-08-13,2005,Portsmouth,Tottenham,0.0,2.0,AwayWin
3,2005-08-13,2005,Middlesbrough,Liverpool,0.0,0.0,Draw
4,2005-08-13,2005,Fulham,Birmingham,0.0,0.0,Draw


## Seção 4 — Split temporal

Divisão do dataset em treino e teste respeitando a ordem cronológica.

> **Por que nunca usar shuffle?** Em séries temporais, embaralhar causa *data leakage*: o modelo veria resultados futuros durante o treino. O corte deve ser sempre por data.

| Conjunto | Temporadas | Jogos aproximados |
|----------|-----------|:-----------------:|
| Treino | 2005/06 → 2021/22 | ~6 460 |
| Teste | 2022/23 → 2025/26 | ~1 520 |

O corte em **2021/22** deixa três temporadas completas para avaliação, representando ~19% dos dados — suficiente para estimar performance out-of-sample sem desperdiçar dados de treino.

Os dados são salvos em formato **Parquet** (comprimido, tipado, rápido de ler) em `dados/`. As colunas de features e Elo serão adicionadas e os parquets re-salvos na Seção 7.

In [4]:
train = df[df["season"] <= 2021].copy()
test  = df[df["season"] >= 2022].copy()

print(f"Treino: {len(train)} jogos (seasons {train['season'].min()}–{train['season'].max()})")
print(f"Teste:  {len(test)} jogos (seasons {test['season'].min()}–{test['season'].max()})")

# Garantir que não há sobreposição temporal
assert train["Date"].max() < test["Date"].min(), "Vazamento temporal detectado!"
print("Corte temporal OK — sem vazamento.")

Treino: 6460 jogos (seasons 2005–2021)
Teste:  1520 jogos (seasons 2022–2025)
Corte temporal OK — sem vazamento.


In [5]:
os.makedirs("dados", exist_ok=True)

train.to_parquet("dados/matches_train.parquet", index=False)
test.to_parquet("dados/matches_test.parquet", index=False)

print("Arquivos salvos:")
print(f"  dados/matches_train.parquet  ({os.path.getsize('dados/matches_train.parquet') / 1024:.1f} KB)")
print(f"  dados/matches_test.parquet   ({os.path.getsize('dados/matches_test.parquet') / 1024:.1f} KB)")

Arquivos salvos:
  dados/matches_train.parquet  (125.2 KB)
  dados/matches_test.parquet   (42.4 KB)


## Seção 6 — Feature Engineering com `build_features(df)`

Geração das variáveis preditoras a partir do histórico de cada time. A função recebe o DataFrame limpo e retorna o mesmo com **29 novas colunas**, agrupadas em quatro categorias:

### Categorias de features

**1. Forma recente — últimos 5 jogos (home e away)**
Gols marcados/sofridos, vitórias, empates, derrotas, saldo de gols, pontos e chutes no alvo dos últimos 5 jogos de cada time, independente de jogar em casa ou fora.

**2. Acumulado na temporada**
Percentual de vitórias, empates e derrotas do time na temporada atual até aquela rodada.

**3. Forma por local**
`home_wins_at_home_last5` — vitórias do mandante nos últimos 5 jogos *em casa*.
`away_wins_away_last5` — vitórias do visitante nos últimos 5 jogos *fora de casa*.

**4. Probabilidades implícitas das odds**
`implied_home_prob`, `implied_draw_prob`, `implied_away_prob` — derivadas das odds da Bet365, normalizadas para somar 1. Capturam a estimativa de mercado sobre o jogo.

---

### Garantia anti-leakage

Toda feature rolling usa `.shift(1)` antes do `.rolling()`, garantindo que **o jogo atual nunca entra no próprio cálculo**. O mesmo vale para o acumulado de temporada via `.expanding()`. NaNs gerados na rodada 1 (sem histórico) são imputados com `0`.

In [ ]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy().sort_values("Date").reset_index(drop=True)

    # ── 1. Long (team-centric) view ──────────────────────────────────────────
    home = pd.DataFrame({
        "match_idx": df.index,
        "Date":      df["Date"],
        "season":    df["season"],
        "team":      df["home_team"],
        "is_home":   1,
        "gf":        df["home_score"].astype(float),
        "ga":        df["away_score"].astype(float),
        "win":       (df["result"] == "HomeWin").astype(float),
        "draw":      (df["result"] == "Draw").astype(float),
        "loss":      (df["result"] == "AwayWin").astype(float),
        "sot":       df["HST"].astype(float),
        "corners":   df["HC"].astype(float),
    })

    away = pd.DataFrame({
        "match_idx": df.index,
        "Date":      df["Date"],
        "season":    df["season"],
        "team":      df["away_team"],
        "is_home":   0,
        "gf":        df["away_score"].astype(float),
        "ga":        df["home_score"].astype(float),
        "win":       (df["result"] == "AwayWin").astype(float),
        "draw":      (df["result"] == "Draw").astype(float),
        "loss":      (df["result"] == "HomeWin").astype(float),
        "sot":       df["AST"].astype(float),
        "corners":   df["AC"].astype(float),
    })

    long = pd.concat([home, away], ignore_index=True)
    long = long.sort_values(["team", "Date"]).reset_index(drop=True)

    # ── 2. Rolling last-5 (todos os jogos, por time) ─────────────────────────
    # Usa transform: nunca perde colunas, preserva alinhamento de índice
    _r5_in  = ["gf",   "ga",  "win",    "draw",   "loss",    "sot",  "corners"]
    _r5_out = ["gf5",  "ga5", "wins5",  "draws5", "losses5", "sot5", "corners5"]

    for col, out in zip(_r5_in, _r5_out):
        long[out] = (
            long.groupby("team")[col]
            .transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
        )

    long["gdiff5"] = long["gf5"] - long["ga5"]
    long["pts5"]   = long["wins5"] * 3 + long["draws5"]

    # ── 3. Acumulado na temporada (até aquela rodada, sem incluir o jogo) ─────
    for col in ["win", "draw", "loss"]:
        long[f"cum_{col}"] = (
            long.groupby(["team", "season"])[col]
            .transform(lambda x: x.shift(1).expanding(min_periods=1).sum())
        )

    cum_total = (long["cum_win"] + long["cum_draw"] + long["cum_loss"]).clip(lower=1)
    long["win_pct"]  = long["cum_win"]  / cum_total
    long["draw_pct"] = long["cum_draw"] / cum_total
    long["loss_pct"] = long["cum_loss"] / cum_total

    no_history = (long["cum_win"] + long["cum_draw"] + long["cum_loss"]) == 0
    long.loc[no_history, ["win_pct", "draw_pct", "loss_pct"]] = np.nan
    long = long.drop(columns=["cum_win", "cum_draw", "cum_loss"])

    # ── 4. Forma por local: últimos 5 em casa / últimos 5 fora ───────────────
    home_long = long[long["is_home"] == 1].copy()
    home_long["wins_at_home5"] = (
        home_long.groupby("team")["win"]
        .transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
    )

    away_long = long[long["is_home"] == 0].copy()
    away_long["wins_away5"] = (
        away_long.groupby("team")["win"]
        .transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
    )

    long = (
        long
        .merge(home_long[["match_idx", "team", "wins_at_home5"]], on=["match_idx", "team"], how="left")
        .merge(away_long[["match_idx", "team", "wins_away5"]],    on=["match_idx", "team"], how="left")
    )

    # ── 5. Pivot de volta ao nível de partida ─────────────────────────────────
    hf = long[long["is_home"] == 1].set_index("match_idx")
    af = long[long["is_home"] == 0].set_index("match_idx")

    feature_map = {
        "home_goals_scored_last5":    hf["gf5"],
        "home_goals_conceded_last5":  hf["ga5"],
        "home_wins_last5":            hf["wins5"],
        "home_draws_last5":           hf["draws5"],
        "home_losses_last5":          hf["losses5"],
        "home_goal_diff_last5":       hf["gdiff5"],
        "home_points_last5":          hf["pts5"],
        "home_shots_on_target_last5": hf["sot5"],
        "home_corners_last5":         hf["corners5"],
        "home_win_pct_season":        hf["win_pct"],
        "home_draw_pct_season":       hf["draw_pct"],
        "home_loss_pct_season":       hf["loss_pct"],
        "home_wins_at_home_last5":    hf["wins_at_home5"],
        "away_goals_scored_last5":    af["gf5"],
        "away_goals_conceded_last5":  af["ga5"],
        "away_wins_last5":            af["wins5"],
        "away_draws_last5":           af["draws5"],
        "away_losses_last5":          af["losses5"],
        "away_goal_diff_last5":       af["gdiff5"],
        "away_points_last5":          af["pts5"],
        "away_shots_on_target_last5": af["sot5"],
        "away_corners_last5":         af["corners5"],
        "away_win_pct_season":        af["win_pct"],
        "away_draw_pct_season":       af["draw_pct"],
        "away_loss_pct_season":       af["loss_pct"],
        "away_wins_away_last5":       af["wins_away5"],
    }

    for name, series in feature_map.items():
        df[name] = series.reindex(df.index).values

    # ── 6. Odds implícitas (imputa nulos com média da temporada) ──────────────
    b365 = df[["B365H", "B365D", "B365A"]].copy()
    for col in ["B365H", "B365D", "B365A"]:
        season_mean = df.groupby("season")[col].transform("mean")
        b365[col] = b365[col].fillna(season_mean)

    total_inv = 1 / b365["B365H"] + 1 / b365["B365D"] + 1 / b365["B365A"]
    df["implied_home_prob"] = (1 / b365["B365H"]) / total_inv
    df["implied_draw_prob"] = (1 / b365["B365D"]) / total_inv
    df["implied_away_prob"] = (1 / b365["B365A"]) / total_inv

    # ── 7. Imputar NaN → 0 (round 1 de cada time não tem histórico) ──────────
    all_feat_cols = list(feature_map.keys()) + [
        "implied_home_prob", "implied_draw_prob", "implied_away_prob"
    ]
    df[all_feat_cols] = df[all_feat_cols].fillna(0)

    return df


print("build_features definida.")

### Aplicação e validação

Chamada de `build_features` no DataFrame completo, seguida de três assertions:
- **Sem NaN** — nenhuma feature pode ter valor faltante após o pipeline
- **Rolling zerado na jornada 1** — todos os times começam 2005/06 sem histórico, portanto todas as rolling features devem ser `0` na primeira jornada
- **Probs implícitas somam 1** — verificação de consistência da normalização das odds

In [ ]:
df_feat = build_features(df)

feat_cols   = [c for c in df_feat.columns if c.endswith(("_last5", "_season", "_prob"))]
rolling_cols = [c for c in feat_cols if c.endswith("_last5")]

# 1. Sem NaN em nenhuma feature
nan_counts = df_feat[feat_cols].isna().sum()
assert nan_counts.sum() == 0, f"NaN encontrados:\n{nan_counts[nan_counts > 0]}"

# 2. Na primeira jornada da temporada (2005-08-13) todos os times têm histórico
#    zerado → todas as rolling features devem ser 0
first_matchday = df_feat["Date"].min()
first_day = df_feat[df_feat["Date"] == first_matchday]
assert (first_day[rolling_cols] == 0).all().all(), "Rolling features não-zero na jornada 1"

# 3. Probabilidades implícitas somam 1 onde odds disponíveis
odds_ok = df_feat["B365H"].notna()
prob_sum = (
    df_feat.loc[odds_ok, "implied_home_prob"]
    + df_feat.loc[odds_ok, "implied_draw_prob"]
    + df_feat.loc[odds_ok, "implied_away_prob"]
)
assert (prob_sum.round(6) == 1.0).all(), "Probabilidades implícitas não somam 1"

print(f"Shape final:               {df_feat.shape}")
print(f"Features geradas:          {len(feat_cols)}")
print(f"Sem NaN:                   OK")
print(f"Rolling zerado na jornada 1: OK")
print(f"Probs implícitas somam 1:  OK")
print()
print(df_feat[feat_cols].describe().T[["mean", "std", "min", "max"]].round(3))

## Seção 7 — Elo Ratings com `build_elo(df)`

O **sistema Elo** é um método de ranqueamento criado pelo físico Arpad Elo para xadrez e amplamente adotado em esportes. A ideia central: cada time tem um rating numérico que sobe quando vence um adversário mais forte e cai quando perde para um mais fraco.

### Fórmula implementada

Elo clássico com **vantagem de mandante** (+100 pontos):

```
E_home = 1 / (1 + 10^((R_away − R_home − 100) / 400))   # expectativa do mandante
E_away = 1 − E_home

S_home = 1.0 (vitória) | 0.5 (empate) | 0.0 (derrota)

R_home_novo = R_home + K × (S_home − E_home)
R_away_novo = R_away + K × (S_away − E_away)
```

### Parâmetros

| Parâmetro | Valor | Significado |
|-----------|------:|-------------|
| `base` | 1500 | Rating inicial de todo time sem histórico |
| `K` | 32 | Sensibilidade — quanto cada jogo move o rating |
| `home_advantage` | 100 | Bônus do mandante na expectativa |

### Colunas geradas

| Coluna | Descrição |
|--------|-----------|
| `elo_home` | Rating do mandante **antes** da partida |
| `elo_away` | Rating do visitante **antes** da partida |
| `elo_diff` | `elo_home − elo_away` — diferença de força entre os times |

### Estratégia de cálculo

O Elo é calculado **uma única vez sobre todo o dataset** (treino + teste em ordem cronológica) antes de separar os conjuntos. Isso garante que o rating do início de 2022/23 reflita o histórico completo até 2021/22, sem reiniciar os valores ao cruzar o corte temporal.

In [ ]:
def build_elo(
    df: pd.DataFrame,
    base: float = 1500,
    K: float = 32,
    home_advantage: float = 100,
) -> pd.DataFrame:
    df = df.copy().sort_values("Date").reset_index(drop=True)

    ratings: dict[str, float] = {}
    elo_home = np.empty(len(df))
    elo_away = np.empty(len(df))

    score_map = {"HomeWin": (1.0, 0.0), "Draw": (0.5, 0.5), "AwayWin": (0.0, 1.0)}

    for i, row in df.iterrows():
        h = row["home_team"]
        a = row["away_team"]

        r_h = ratings.get(h, base)
        r_a = ratings.get(a, base)

        # Registrar rating PRÉ-partida
        elo_home[i] = r_h
        elo_away[i] = r_a

        # Expectativa com vantagem de mandante
        e_h = 1 / (1 + 10 ** ((r_a - r_h - home_advantage) / 400))
        e_a = 1 - e_h

        s_h, s_a = score_map[row["result"]]

        # Atualizar após registrar
        ratings[h] = r_h + K * (s_h - e_h)
        ratings[a] = r_a + K * (s_a - e_a)

    df["elo_home"] = elo_home
    df["elo_away"] = elo_away
    df["elo_diff"] = df["elo_home"] - df["elo_away"]

    return df, ratings


print("build_elo definida.")

### Aplicação, re-split e sanity check

O Elo é aplicado sobre `df_feat` (já com todas as features da Seção 6) e retorna dois valores: o DataFrame enriquecido e o dicionário `final_ratings` com o rating atual de cada time.

Em seguida, o dataset é re-separado pelo mesmo corte temporal da Seção 4, os parquets são sobrescritos com as 3 colunas adicionais e um **sanity check** imprime o ranking dos times ao final do período de treino.

> Times como Man City, Liverpool e Chelsea no topo — e times rebaixados no fundo — indicam que o sistema está capturando a hierarquia histórica da liga corretamente.

In [ ]:
# Aplicar Elo sobre o DataFrame completo (treino + teste) para rating contínuo
df_elo, final_ratings = build_elo(df_feat)

# Re-separar pelo corte temporal
train_final = df_elo[df_elo["season"] <= 2021].copy()
test_final  = df_elo[df_elo["season"] >= 2022].copy()

# Confirmar colunas presentes
assert {"elo_home", "elo_away", "elo_diff"}.issubset(train_final.columns)
assert {"elo_home", "elo_away", "elo_diff"}.issubset(test_final.columns)

# Salvar parquets atualizados
train_final.to_parquet("dados/matches_train.parquet", index=False)
test_final.to_parquet("dados/matches_test.parquet", index=False)

print(f"elo_home / elo_away / elo_diff presentes em treino e teste: OK")
print(f"Treino: {len(train_final)} jogos | Teste: {len(test_final)} jogos")
print(f"Range elo_diff treino: [{train_final['elo_diff'].min():.1f}, {train_final['elo_diff'].max():.1f}]")
print()

# Sanity check: top 5 e bottom 5 ao final do período de treino
# Filtrar apenas times que jogaram na última temporada de treino (2021/22)
last_train_season = df_elo[df_elo["season"] == 2021]
teams_in_last = set(last_train_season["home_team"]) | set(last_train_season["away_team"])
ratings_last = {t: final_ratings[t] for t in teams_in_last if t in final_ratings}

# Recalcular ratings ao final de 2021/22 (última partida de treino)
cutoff_date = train_final["Date"].max()
df_train_elo = df_elo[df_elo["Date"] <= cutoff_date]

# Pegar rating pré-partida da ÚLTIMA partida de cada time no período de treino
last_home = df_train_elo.groupby("home_team").apply(lambda g: g.nlargest(1, "Date")[["Date", "elo_home"]].iloc[0])
last_away = df_train_elo.groupby("away_team").apply(lambda g: g.nlargest(1, "Date")[["Date", "elo_away"]].iloc[0])

home_r = last_home["elo_home"].rename("rating")
home_r.index.name = "team"
away_r = last_away["elo_away"].rename("rating")
away_r.index.name = "team"

end_ratings = pd.concat([home_r, away_r]).groupby(level=0).max().sort_values(ascending=False)

print("=== Top 5 times por Elo (fim do treino) ===")
print(end_ratings.head(5).round(1).to_string())
print()
print("=== Bottom 5 times por Elo (fim do treino) ===")
print(end_ratings.tail(5).round(1).to_string())

## Seção 8 — Verificação de Leakage

Antes de qualquer imputação, toda feature rolling deve ser `NaN` no primeiro jogo de cada time (sem histórico anterior). Esta seção reconstrói a lógica de cálculo de forma independente — sem chamar `build_features` — e verifica isso via assert.

Dois invariantes são checados:

| Invariante | O que garante |
|------------|--------------|
| Rolling last-5: 1º jogo de cada time no dataset | Time sem histórico algum não pode ter valor rolling não-nulo |
| Acumulado de temporada: 1º jogo de cada (time, temporada) | Stat de temporada reseta a cada season — 1ª rodada não tem jogos anteriores |

In [ ]:
# Reconstrói long format apenas com o necessário para a verificação
_home = pd.DataFrame({
    "team":   df["home_team"],
    "Date":   df["Date"],
    "season": df["season"],
    "win":    (df["result"] == "HomeWin").astype(float),
})
_away = pd.DataFrame({
    "team":   df["away_team"],
    "Date":   df["Date"],
    "season": df["season"],
    "win":    (df["result"] == "AwayWin").astype(float),
})
lv = pd.concat([_home, _away], ignore_index=True)

# ── Invariante 1: rolling last-5 ─────────────────────────────────────────────
lv_team = lv.sort_values(["team", "Date"]).reset_index(drop=True)
lv_team["rolling_pre"] = (
    lv_team.groupby("team")["win"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
)
first_per_team = lv_team.groupby("team").head(1).index
assert lv_team.loc[first_per_team, "rolling_pre"].isna().all(), \
    "Leakage: rolling não-NaN no 1º jogo de algum time"

n_teams = lv_team["team"].nunique()
print(f"Rolling last-5   : 1º jogo de cada um dos {n_teams} times tem NaN antes de imputar — OK")

# ── Invariante 2: acumulado de temporada ─────────────────────────────────────
lv_season = lv.sort_values(["team", "season", "Date"]).reset_index(drop=True)
lv_season["cum_pre"] = (
    lv_season.groupby(["team", "season"])["win"]
    .transform(lambda x: x.shift(1).expanding(min_periods=1).sum())
)
first_per_team_season = lv_season.groupby(["team", "season"]).head(1).index
assert lv_season.loc[first_per_team_season, "cum_pre"].isna().all(), \
    "Leakage: acumulado-temporada não-NaN no 1º jogo de algum (time, temporada)"

n_pairs = lv_season.groupby(["team", "season"]).ngroups
print(f"Acumulado/season : 1º jogo de cada um dos {n_pairs} pares (time, temporada) tem NaN antes de imputar — OK")

## Seção 9 — Salvar Parquets Finais

`build_features()` é aplicada **separadamente** em treino e teste — validando que a função funciona como unidade independente em qualquer subconjunto do DataFrame.

> O split temporal já foi definido na Seção 4, antes de qualquer cálculo de feature. Nenhuma stat do jogo atual entra como feature — todo cálculo usa apenas jogos anteriores via `shift(1)`.

Os arquivos gerados são os que serão consumidos pelo pipeline de modelagem:

| Arquivo | Conteúdo |
|---------|----------|
| `dados/feature_matrix_train.parquet` | Treino com todas as features (2005/06 → 2021/22) |
| `dados/feature_matrix_test.parquet` | Teste com todas as features (2022/23 → 2025/26) |

In [ ]:
train_fm = build_features(train)
test_fm  = build_features(test)

feat_cols = [c for c in train_fm.columns if c.endswith(("_last5", "_season", "_prob"))]

# Confirmar ausência de NaN após imputação
assert train_fm[feat_cols].isna().sum().sum() == 0, "NaN em feature_matrix_train"
assert test_fm[feat_cols].isna().sum().sum() == 0,  "NaN em feature_matrix_test"

os.makedirs("dados", exist_ok=True)
train_fm.to_parquet("dados/feature_matrix_train.parquet", engine="pyarrow", index=False)
test_fm.to_parquet("dados/feature_matrix_test.parquet",  engine="pyarrow", index=False)

print(f"feature_matrix_train : {train_fm.shape}  →  dados/feature_matrix_train.parquet")
print(f"feature_matrix_test  : {test_fm.shape}  →  dados/feature_matrix_test.parquet")
print(f"Sem NaN nas {len(feat_cols)} colunas de feature: OK")

## Seção 10 — Features Avançadas com `build_advanced_features(train_df, test_df)`

Expansão do pipeline com features derivadas das já existentes e uma feature que requer acesso exclusivo ao treino.

### Novas features

| Grupo | Colunas | Descrição |
|-------|---------|-----------|
| **Vantagem relativa** | `goals_scored_adv`, `goals_conceded_adv`, `points_adv`, `goal_diff_adv`, `win_pct_adv`, `shots_adv` | Diferença home − away das principais stats de forma. Captura o desequilíbrio de força entre os times no contexto do confronto |
| **Fase da temporada** | `round_norm` | Rodada implícita (nº de jogos disputados pelo time na season) ÷ 38. Valores perto de 0 = início de temporada; perto de 1 = final |
| **Pontos por jogo** | `home_ppg_season`, `away_ppg_season`, `ppg_adv` | Estimativa de PPG acumulado (win × 3 + draw) sobre o percentual de resultados da temporada |
| **Eficiência ofensiva** | `home_goal_eff`, `away_goal_eff`, `goal_eff_adv` | Proporção de gols marcados sobre o total (marcados + sofridos + 0.5), escalada entre 0 e 1 |
| **Taxa histórica por time** | `home_team_hist_wr`, `away_team_hist_wr` | % de vitórias em casa / fora de casa por time, calculada **exclusivamente sobre o treino** e aplicada por mapeamento no teste |

> **Anti-leakage nas taxas históricas:** o mapeamento é gerado antes de chamar `_add()` sobre o teste, usando apenas `train_df`. Times do teste que nunca aparecem no treino recebem a média global do treino como fallback.

In [ ]:
def build_advanced_features(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:

    # ── Taxas históricas calculadas APENAS no treino ──────────────────────────
    hist_home_wr = (
        train_df.groupby("home_team")["result"]
        .apply(lambda x: (x == "HomeWin").mean())
    )
    hist_away_wr = (
        train_df.groupby("away_team")["result"]
        .apply(lambda x: (x == "AwayWin").mean())
    )
    global_home_wr = float((train_df["result"] == "HomeWin").mean())
    global_away_wr = float((train_df["result"] == "AwayWin").mean())

    def _add(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy().sort_values("Date").reset_index(drop=True)

        # 1. Vantagem relativa (home − away)
        df["goals_scored_adv"]   = df["home_goals_scored_last5"]    - df["away_goals_scored_last5"]
        df["goals_conceded_adv"] = df["away_goals_conceded_last5"]  - df["home_goals_conceded_last5"]
        df["points_adv"]         = df["home_points_last5"]          - df["away_points_last5"]
        df["goal_diff_adv"]      = df["home_goal_diff_last5"]       - df["away_goal_diff_last5"]
        df["win_pct_adv"]        = df["home_win_pct_season"]        - df["away_win_pct_season"]
        df["shots_adv"]          = df["home_shots_on_target_last5"] - df["away_shots_on_target_last5"]

        # 2. round_norm — rodada implícita por contagem de jogos na temporada
        if "round_norm" not in df.columns:
            home_r = (
                df[["Date", "season", "home_team"]]
                .rename(columns={"home_team": "team"})
                .assign(is_home=1, match_idx=df.index)
            )
            away_r = (
                df[["Date", "season", "away_team"]]
                .rename(columns={"away_team": "team"})
                .assign(is_home=0, match_idx=df.index)
            )
            long_r = pd.concat([home_r, away_r]).sort_values(["team", "season", "Date"])
            long_r["game_num"] = long_r.groupby(["team", "season"]).cumcount() + 1

            h_gnum = long_r[long_r["is_home"] == 1].set_index("match_idx")["game_num"]
            a_gnum = long_r[long_r["is_home"] == 0].set_index("match_idx")["game_num"]
            df["round_norm"] = ((h_gnum.reindex(df.index) + a_gnum.reindex(df.index)) / 2) / 38

        # 3. Pontos por jogo estimado na temporada
        df["home_ppg_season"] = df["home_win_pct_season"] * 3 + df["home_draw_pct_season"]
        df["away_ppg_season"] = df["away_win_pct_season"] * 3 + df["away_draw_pct_season"]
        df["ppg_adv"]         = df["home_ppg_season"] - df["away_ppg_season"]

        # 4. Eficiência ofensiva (ratio escalado entre 0 e 1)
        df["home_goal_eff"] = df["home_goals_scored_last5"] / (
            df["home_goals_scored_last5"] + df["home_goals_conceded_last5"] + 0.5
        )
        df["away_goal_eff"] = df["away_goals_scored_last5"] / (
            df["away_goals_scored_last5"] + df["away_goals_conceded_last5"] + 0.5
        )
        df["goal_eff_adv"] = df["home_goal_eff"] - df["away_goal_eff"]

        # 5. Taxa histórica por time — mapa calculado no treino, aplicado aqui
        df["home_team_hist_wr"] = df["home_team"].map(hist_home_wr).fillna(global_home_wr)
        df["away_team_hist_wr"] = df["away_team"].map(hist_away_wr).fillna(global_away_wr)

        return df

    return _add(train_df), _add(test_df)


print("build_advanced_features definida.")

### Aplicação e re-gravação dos parquets

Aplicada sobre `train_final` e `test_final` (DataFrames com todas as features + Elo da Seção 7), que têm rolling calculado sobre o dataset completo — sem cold-start no conjunto de teste.

In [ ]:
train_adv, test_adv = build_advanced_features(train_final, test_final)

new_feat_cols = [
    "goals_scored_adv", "goals_conceded_adv", "points_adv", "goal_diff_adv",
    "win_pct_adv", "shots_adv",
    "round_norm",
    "home_ppg_season", "away_ppg_season", "ppg_adv",
    "home_goal_eff", "away_goal_eff", "goal_eff_adv",
    "home_team_hist_wr", "away_team_hist_wr",
]

assert train_adv[new_feat_cols].isna().sum().sum() == 0, "NaN em features novas (treino)"
assert test_adv[new_feat_cols].isna().sum().sum() == 0,  "NaN em features novas (teste)"

os.makedirs("dados", exist_ok=True)
train_adv.to_parquet("dados/feature_matrix_train.parquet", engine="pyarrow", index=False)
test_adv.to_parquet("dados/feature_matrix_test.parquet",  engine="pyarrow", index=False)

print(f"feature_matrix_train : {train_adv.shape}")
print(f"feature_matrix_test  : {test_adv.shape}")
print(f"Sem NaN nas {len(new_feat_cols)} novas features: OK")
print()
print(train_adv[new_feat_cols].describe().T[["mean", "std", "min", "max"]].round(3))

## Seção 11 — Pairplot das 10 Features Mais Relevantes

Visualização das relações entre as 10 features com maior importância por permutation importance no modelo treinado. A cor representa o resultado da partida.

In [ ]:
TOP10_FEATURES = [
    "implied_away_prob",
    "implied_home_prob",
    "away_shots_on_target_last5",
    "away_loss_pct_season",
    "goal_eff_adv",
    "away_goals_scored_last5",
    "goals_conceded_adv",
    "away_draw_pct_season",
    "goals_scored_adv",
    "away_goal_eff",
]

plot_df = train_adv[TOP10_FEATURES + ["result"]].copy()

g = sns.pairplot(
    plot_df,
    hue="result",
    hue_order=["HomeWin", "Draw", "AwayWin"],
    palette={"HomeWin": "#2196F3", "Draw": "#9E9E9E", "AwayWin": "#F44336"},
    plot_kws={"alpha": 0.25, "s": 8},
    diag_kind="kde",
    corner=True,
)
g.figure.suptitle("Pairplot — Top 10 Features (treino)", y=1.01, fontsize=13)
plt.tight_layout()
plt.savefig("pairplot_top10.png", dpi=100, bbox_inches="tight")
plt.show()
print("Salvo em pairplot_top10.png")